In [1]:
from pathlib import Path

import polars as pl
import scipy as sp

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dqdvs import dqdv_histogram

from config import DATA_PATH
ZENODO_DATA_PATH = Path(DATA_PATH) / Path("HALF_CELL_OCVS_ZENODO")

In [2]:
################################ LOAD DATA  ###################################
all_filepaths = ZENODO_DATA_PATH.rglob("*.parquet")
#[filepath.name for filepath in all_filepaths if ("silicongraphite" in filepath.stem) and ("20250405" in filepath.stem) and ("p-ocvhold" in filepath.stem)]

In [22]:
filename = 'sintef__sintef-lnmo-R2032-intelligent1-32a68c__20250424__p-ocvhold__RT.bdf.parquet'
BIN_SIZE = 1.5e-3

df = pl.read_parquet(ZENODO_DATA_PATH / Path(filename))

df_subset = (df
            .filter(pl.col('Step Index / 1')==2)
            .filter(pl.col('Cycle Count / 1')==3)
            )

q = df_subset['Cumulative Capacity / Ah'].to_numpy()
v = df_subset['Voltage / V'].to_numpy()

q = q/q.max() #Normalized capacity
v_dqdv, dqdv = dqdv_histogram(q=q, v=v, bin_size=BIN_SIZE)
q_reconstructed = sp.integrate.cumulative_trapezoid(dqdv, v_dqdv, initial=0.0) + q[0]

In [4]:
df.columns

['Test Time / s',
 'Unix Time / s',
 'Current / A',
 'Voltage / V',
 'Cycle Count / 1',
 'Step Index / 1',
 'Cumulative Capacity / Ah']

In [69]:
COLOR1 = "#7F7F7F"
COLOR2 = "#00CC96"

fig = make_subplots(rows=1, 
                    cols=2, 
                    shared_yaxes=True, 
                    horizontal_spacing=0.01,
                    column_widths=[0.55, 0.45],
                    #subplot_titles=[x for pair in zip(cell_names.keys(), [""]*len(cell_names)) for x in pair]
                    )

fig.add_trace( go.Scatter(x=q, 
                        y=v, 
                        name="Original", 
                        mode="markers",
                        #line=dict(color=COLOR1, width=6),
                        marker=dict(color=COLOR1, size=12, opacity=0.1), 
                        showlegend=False), 
                        row=1, col=1 )

# fig.add_trace( go.Scatter(x=dqdv, 
#                         y=v_dqdv, 
#                         name="Incremental Capacity", 
#                         line=dict(color="Red", width=2), 
#                         showlegend=False), 
#                         row=1, col=2 )

fig.add_trace(go.Bar(x=dqdv, 
                     y=v_dqdv, 
                     name="Bins", 
                     orientation="h", 
                     opacity=0.9, 
                     marker=dict(color=COLOR2), 
                     showlegend=False), 
                    row=1, col=2 )

fig.add_trace( go.Scatter(x=q_reconstructed, 
                        y=v_dqdv, 
                        name="Reconstruction", 
                        line=dict(color=COLOR2, width=5), 
                        showlegend=False), 
                        row=1, col=1 )

fig.update_layout(
    template="ggplot2",
    margin=dict(l=10, r=10, t=10, b=10),
    width=800, 
    height=500,
    font=dict(size=24),
    legend=dict(
        orientation="h",    # horizontal
        yanchor="bottom",   # anchor legend's bottom
        y=1.02,             # place above the plot area
        xanchor="center",
        x=0.5
    ),)

fig.update_xaxes(title_text="Cumulative Capacity", ticks="", row=1, col=1, showgrid=False, showticklabels=False, range=[0.27, 1.005])
fig.update_xaxes(title_text="Incremental Capacity", ticks="", row=1, col=2,  showticklabels=False, showgrid=False, range=[0, 56])

fig.update_yaxes(title_text="Voltage", ticks="", row=1, col=1, showgrid=False, range=[4.66, 4.78], showticklabels=False)
fig.update_yaxes(title_text="", ticks="", row=1, col=2, showgrid=False, range=[4.66, 4.78], showticklabels=False)

fig.add_annotation(x=0.28, 
                    y=4.770,
                    text="— Original Voltage Curve",
                    showarrow=False,
                    font=dict(size=23,  color=COLOR1, weight=1000), 
                    xanchor="left",
                    row=1, col=1 )

fig.add_annotation(x=0.28, 
                    y=4.760,
                    text="— Reconstruction from IC",
                    showarrow=False,
                    font=dict(size=23,  color=COLOR2, weight=1000), 
                    xanchor="left",
                    row=1, col=1 )

fig.show()

In [ ]:
fig.write_image("figures/toc.png", scale=10)